# KhaledDSML 16 — Research integrity & scientific diagnostics
## النزاهة البحثية وتدقيق جودة البيانات

**Synthetic examples only / بيانات اصطناعية فقط.** Development version v0.10.0rc1; the stable version published on PyPI is 0.9.0.

**Scope:** McNemar, MAPE, GRIM, t-p recomputation, schema contracts, Page–Hinkley. This is not a fraud detection or causal inference system.

In [ ]:
%pip install -q "git+https://github.com/noble2013/KhaledDSML.git@main"

## 1. McNemar paired classifiers | مقارنة نموذجين على نفس الحالات

يجب عدم استخدامه لمجموعتي اختبار مستقلتين. / Do not compare independent holdout sets as though they are paired.

In [ ]:
import numpy as np
import pandas as pd
from khaled_dsml import (
    mcnemar_paired_predictions, mean_absolute_percentage_error_strict,
    grim_mean_consistency, recalculate_t_test_p, validate_dataframe_contract,
    page_hinkley_change_monitor, display_analysis_dashboard,
)
y_true = np.array([0,1]*50)
a_correct = np.array([True]*55+[True]*20+[False]*5+[False]*20)
b_correct = np.array([True]*55+[False]*20+[True]*5+[False]*20)
pred_a = np.where(a_correct,y_true,1-y_true)
pred_b = np.where(b_correct,y_true,1-y_true)
display_analysis_dashboard(
    mcnemar_paired_predictions(y_true,pred_a,pred_b),theme="amethyst"
)

## 2. MAPE | الخطأ النسبي للتنبؤ

القيم الحقيقية الصفرية ممنوعة صراحةً، ولا يصح اعتبار MAPE اختبار دلالة. / Zero denominators fail explicitly; MAPE is descriptive.

In [ ]:
display_analysis_dashboard(
    mean_absolute_percentage_error_strict([10,20,30,40],[11,18,33,36]),
    theme="ocean"
)

## 3. GRIM arithmetic | التدقيق الحسابي للمتوسطات

التناسق الحسابي ليس إثباتًا لصحة البيانات، وعدم الاتساق ليس دليلًا على تزوير. / Inconsistency is a flag for verification, never proof of wrongdoing.

In [ ]:
display_analysis_dashboard(
    grim_mean_consistency("3.21",7,decimals=2,minimum_score=1,maximum_score=5),
    theme="aurora"
)

## 4. Recomputed t p-value | إعادة حساب قيمة p

راجع اتجاه الفرضية ودرجات الحرية وقاعدة التقريب المنشورة. / Verify hypothesis tail, df and actual article rounding.

In [ ]:
p_audit = recalculate_t_test_p(2.3,20,alternative="two-sided",reported_p="0.900")
display_analysis_dashboard(p_audit,theme="amethyst")

## 5. Data contract | قواعد بنية البيانات

لا تعرض النتيجة قيم الصفوف، بل أعداد المخالفات فقط. / Audit aggregate violations only.

In [ ]:
frame = pd.DataFrame({"id":[1,2,2,4],"income":[100,200,-10,np.nan],"arm":["A","B","X","A"]})
contract = {"id":{"kind":"integer","unique":True,"nullable":False},
            "income":{"kind":"numeric","min":0,"nullable":False},
            "arm":{"kind":"string","allowed_values":["A","B"]}}
display_analysis_dashboard(validate_dataframe_contract(frame,contract),theme="ocean")

## 6. Page–Hinkley | إنذار التغيرات المتتابعة

الإنذارات ليست قيم p ولا تضمن ضبط الإنذارات الكاذبة دون معايرة. / Alarms are descriptive and require a calibrated threshold.

In [ ]:
samples = np.r_[np.zeros(100),np.ones(120)*4]
result = page_hinkley_change_monitor(samples,delta=.05,threshold=10,
                                    min_instances=15,direction="up")
display_analysis_dashboard(result,theme="aurora")

## Research integrity checklist | قائمة النزاهة العلمية

- Separate training and held-out validation before comparing predictions.
- Record score scales, n, rounding conventions and alternatives before auditing published findings.
- Declare schema contract rules in advance; don't tune them to conceal violations.
- Calibrate Page–Hinkley on reference data; account for repeated monitoring and autocorrelation.

الاستنتاجات في هذا الدفتر توضيحية ولا تشكل أحكامًا بشأن بحوث أو أشخاص حقيقيين.